# Demo 3 — Graph Neural Network Attacks
## Nettack on the Cora Citation Network

**Threat Model:** White-box targeted attack

**Goal:** Make a GCN misclassify a *specific target node* by adding/removing
a small number of edges or node features.

**Library:** `deeprobust`

In [ ]:
# 1. Create data directory to prevent FileNotFoundError when downloading Cora
import os
os.makedirs('./data', exist_ok=True)

# 2. Install dependencies (Uncomment to install)
# !pip install deeprobust torch scipy numpy torch_geometric

# 3. Install pre-compiled PyG binaries to prevent hanging on source compilation
# import torch
# pt_version = torch.__version__
# !pip install torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-{pt_version}.html


In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from deeprobust.graph.data import Dataset
from deeprobust.graph.defense import GCN
from deeprobust.graph.targeted_attack import Nettack
torch.manual_seed(42); np.random.seed(42)
print('DeepRobust ready.')

## Step 1 — Load the Cora Dataset

Cora: **2708 nodes** (papers), **5429 edges** (citations), **7 classes** (topics).
Each node has a **1433-dimensional** bag-of-words feature vector.

In [ ]:
data = Dataset(root='./data', name='cora', setting='nettack')
adj, features, labels = data.adj, data.features, data.labels
idx_train, idx_val, idx_test = data.idx_train, data.idx_val, data.idx_test
print(f'Nodes   : {adj.shape[0]}')
print(f'Edges   : {adj.nnz // 2}')
print(f'Features: {features.shape[1]}')
print(f'Classes : {labels.max() + 1}')

## Step 2 — Train the Victim GCN

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
gcn = GCN(nfeat=features.shape[1], nclass=labels.max()+1, nhid=16,
          dropout=0.5, with_relu=True, with_bias=True, device=device).to(device)
gcn.fit(features, adj, labels, idx_train, idx_val, verbose=False)
out = gcn.predict()
clean_acc = (out.argmax(1)[idx_test].cpu().numpy() == labels[idx_test]).mean()
print(f'Clean GCN accuracy on test set: {clean_acc*100:.2f}%')

## Step 3 — Select Target Node and Run Nettack

### How Nettack Works
1. Compute the **linearised approximation** of the GCN's loss
2. Greedily select the edge/feature perturbation maximising loss for the target
3. Repeat for `n_perturbations` steps

Key insight: attacking graph structure has a *global* effect through neighbourhood aggregation.

In [ ]:
correct_mask = (out.argmax(1)[idx_test].cpu().numpy() == labels[idx_test])
correct_test_nodes = idx_test[correct_mask]
target_node = int(correct_test_nodes[0])
true_label  = labels[target_node]
clean_pred  = out.argmax(1)[target_node].item()
clean_pred_probs = out[target_node].detach().cpu().numpy()
print(f'Target node: {target_node} | True label: {true_label} | Clean pred: {clean_pred}')

import matplotlib.pyplot as plt
def plot_probs(probs, title, filename):
    fig, ax = plt.subplots(figsize=(5, 3))
    ax.bar(range(len(probs)), np.exp(probs), color='#3498db')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Class'); ax.set_ylabel('Probability')
    ax.set_xticks(range(len(probs)))
    plt.tight_layout()
    plt.savefig(filename, dpi=150)
    plt.close()
plot_probs(clean_pred_probs, 'Before Attack: Clean Node Probabilities', '03_before_attack.png')

attacker = Nettack(gcn, nnodes=adj.shape[0], attack_structure=True,
                   attack_features=True, device=device).to(device)
attacker.attack(features, adj, labels, target_node, 2, verbose=False)
gcn_mid = GCN(nfeat=features.shape[1], nclass=labels.max()+1, nhid=16,
           dropout=0.5, with_relu=True, with_bias=True, device=device).to(device)
gcn_mid.fit(attacker.modified_features, attacker.modified_adj, labels, idx_train, idx_val, verbose=False)
out_mid = gcn_mid.predict()
plot_probs(out_mid[target_node].detach().cpu().numpy(), 'In Process: After 2 Perturbations', '03_in_process_attack.png')

attacker.attack(features, adj, labels, target_node, 5, verbose=False)
print(f'Nettack made 5 perturbations.')

## Step 4 — Evaluate on the Perturbed Graph

In [ ]:
gcn2 = GCN(nfeat=features.shape[1], nclass=labels.max()+1, nhid=16,
           dropout=0.5, with_relu=True, with_bias=True, device=device).to(device)
gcn2.fit(attacker.modified_features, attacker.modified_adj, labels,
         idx_train, idx_val, verbose=False)
out2 = gcn2.predict()
plot_probs(out2[target_node].detach().cpu().numpy(), 'After Attack: After 5 Perturbations', '03_after_attack.png')
attacked_pred = out2.argmax(1)[target_node].item()
overall_acc   = (out2.argmax(1)[idx_test].cpu().numpy() == labels[idx_test]).mean()
print(f'Before attack: predicted {clean_pred} | After attack: predicted {attacked_pred}')
print(f'True label: {true_label}')
print(f'Overall test accuracy after attack: {overall_acc*100:.2f}%')

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
cats = ['Clean Accuracy', 'After Attack']
vals = [clean_acc*100, overall_acc*100]
colors = ['#27ae60', '#e74c3c']
bars = axes[0].bar(cats, vals, color=colors, edgecolor='black', width=0.4)
axes[0].bar_label(bars, fmt='%.1f%%', padding=3)
axes[0].set_ylim(0, 110)
axes[0].set_title('Overall GCN Accuracy', fontweight='bold')
preds_vs = [clean_pred, attacked_pred]
bar_c    = ['#27ae60' if p==true_label else '#e74c3c' for p in preds_vs]
axes[1].bar(['Before Attack', 'After Attack'], [1, 1], color=bar_c, edgecolor='black', width=0.4)
axes[1].set_yticks([])
for i, p in enumerate(preds_vs):
    ok = '✅' if p == true_label else '❌ FOOLED'
    axes[1].text(i, 0.5, f'Pred={p}\n{ok}', ha='center', va='center',
                 fontsize=11, color='white', fontweight='bold')
axes[1].set_title(f'Target Node {target_node} (True={true_label})', fontweight='bold')
plt.suptitle('Nettack on Cora GCN', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('03_graph_attack_results.png', dpi=150)
plt.show()

import networkx as nx

def get_ego_edges(adj_matrix, node, hops=2):
    try:
        G = nx.from_scipy_sparse_array(adj_matrix)
    except:
        G = nx.from_scipy_sparse_matrix(adj_matrix)
    ego = nx.ego_graph(G, node, radius=hops)
    return set(ego.edges())

clean_edges = get_ego_edges(adj, target_node)
perturbed_edges = get_ego_edges(attacker.modified_adj, target_node)

added_edges = perturbed_edges - clean_edges
removed_edges = clean_edges - perturbed_edges

# Create a combined graph of BOTH clean and perturbed ego networks
# This ensures nodes that get completely disconnected by edge removal
# are still present in the layout positions (fixing KeyError).
try:
    G_clean = nx.from_scipy_sparse_array(adj)
    G_pert = nx.from_scipy_sparse_array(attacker.modified_adj)
except:
    G_clean = nx.from_scipy_sparse_matrix(adj)
    G_pert = nx.from_scipy_sparse_matrix(attacker.modified_adj)

ego_clean = nx.ego_graph(G_clean, target_node, radius=2)
ego_pert = nx.ego_graph(G_pert, target_node, radius=2)

# Combine them for calculating positions and drawing all nodes
G_combined = nx.compose(ego_clean, ego_pert)

pos = nx.spring_layout(G_combined, seed=42)
plt.figure(figsize=(10, 8))

# Draw nodes
nx.draw_networkx_nodes(G_combined, pos, node_color='lightgray', node_size=100)
nx.draw_networkx_nodes(G_combined, pos, nodelist=[target_node], node_color='red', node_size=300, label='Target Node')

# Normal edges are those that were in the clean graph and weren't removed
normal_edges = [e for e in ego_clean.edges() if e not in removed_edges and (e[1], e[0]) not in removed_edges]

# Draw edges
nx.draw_networkx_edges(G_combined, pos, edgelist=normal_edges, edge_color='gray', alpha=0.5)
if added_edges:
    nx.draw_networkx_edges(G_combined, pos, edgelist=list(added_edges), edge_color='red', width=2.5, style='dashed', label='Added by Attacker')
if removed_edges:
    nx.draw_networkx_edges(G_combined, pos, edgelist=list(removed_edges), edge_color='blue', width=2.5, style='dotted', label='Removed by Attacker')

plt.title('2-Hop Ego Graph Around Target Node (Perturbations Highlighted)', fontsize=14, fontweight='bold')
plt.legend(loc='best')
plt.axis('off')
plt.tight_layout()
plt.savefig('03_graph_perturbation.png', dpi=150)
plt.show()


In [ ]:
import networkx as nx

def get_ego_edges(adj_matrix, node, hops=2):
    try:
        # newer networkx
        G = nx.from_scipy_sparse_array(adj_matrix)
    except:
        # older networkx
        G = nx.from_scipy_sparse_matrix(adj_matrix)
    ego = nx.ego_graph(G, node, radius=hops)
    return set(ego.edges())

clean_edges = get_ego_edges(adj, target_node)
perturbed_edges = get_ego_edges(attacker.modified_adj, target_node)

added_edges = perturbed_edges - clean_edges
removed_edges = clean_edges - perturbed_edges

try:
    G_show = nx.from_scipy_sparse_array(attacker.modified_adj)
except:
    G_show = nx.from_scipy_sparse_matrix(attacker.modified_adj)
ego_G = nx.ego_graph(G_show, target_node, radius=2)

pos = nx.spring_layout(ego_G, seed=42)
plt.figure(figsize=(10, 8))

# Draw base ego graph
nx.draw_networkx_nodes(ego_G, pos, node_color='lightgray', node_size=100)
nx.draw_networkx_nodes(ego_G, pos, nodelist=[target_node], node_color='red', node_size=300, label='Target Node')

# Draw edges
nx.draw_networkx_edges(ego_G, pos, edgelist=[e for e in ego_G.edges() if e not in added_edges], edge_color='gray', alpha=0.5)
nx.draw_networkx_edges(ego_G, pos, edgelist=list(added_edges), edge_color='red', width=2.5, style='dashed', label='Added by Attacker')
if removed_edges:
    nx.draw_networkx_edges(ego_G, pos, edgelist=list(removed_edges), edge_color='blue', width=2.5, style='dotted', label='Removed by Attacker')

plt.title('2-Hop Ego Graph Around Target Node (Perturbations Highlighted)', fontsize=14, fontweight='bold')
plt.legend(loc='best')
plt.axis('off')
plt.tight_layout()
plt.savefig('03_graph_perturbation.png', dpi=150)
plt.show()
